# The Vanished Clause

Reconstruct a CFR section's wording *before* an amendment, given how it reads *after*.

Three progressive submissions:

1. **Copy the input through** — the honest floor. Scores ~0 by construction.
2. **Substitution table + sentence locator** (CPU) — mine what amendments do to this
   language from train, find the likeliest amended sentence, revert one edit.
3. **Seq2seq over the amended passage** (GPU) — condition a pretrained encoder-decoder
   on the located passage and generate its prior wording.

The metric scores only the tokens the amendment removed, balanced against what the
prediction adds. That shapes every choice below: **editing text the amendment never
touched costs score**, so a good solution makes few, confident changes.


In [ ]:
import os, re, json
from collections import Counter, defaultdict
import pandas as pd

SEED = 20260925
DATA = next(p for p in ("/data", "dataset/public",
                        "/kaggle/input/ecfr-amendment-reversal/public")
            if os.path.exists(os.path.join(p, "test.csv")))
OUT = "./working" if os.path.isdir("./working") else "."
os.makedirs(OUT, exist_ok=True)

train = pd.read_csv(os.path.join(DATA, "train.csv"))
test  = pd.read_csv(os.path.join(DATA, "test.csv"))
print("train", train.shape, "| test", test.shape)
train.head(2)

## How big is a change?

This is the single most important number in the task: it decides what the metric
has to look like, and what a solution should try to do.

In [ ]:
TOKEN = re.compile(r"\w+|[^\w\s]")
tokenize = lambda t: TOKEN.findall(t or "")

def changed_positions(a_toks, b_toks):
    """Indices of b_toks not matched through from a_toks (LCS backtrace)."""
    n, m = len(a_toks), len(b_toks)
    if m == 0: return set()
    if n == 0: return set(range(m))
    prev = [0]*(m+1); choices = []
    for i in range(1, n+1):
        cur = [0]*(m+1); row = bytearray(m+1); ai = a_toks[i-1]
        for j in range(1, m+1):
            if ai == b_toks[j-1]: cur[j] = prev[j-1]+1; row[j] = 1
            elif prev[j] >= cur[j-1]: cur[j] = prev[j]; row[j] = 2
            else: cur[j] = cur[j-1]; row[j] = 3
        choices.append(row); prev = cur
    matched = set(); i, j = n, m
    while i > 0 and j > 0:
        c = choices[i-1][j]
        if c == 1: matched.add(j-1); i -= 1; j -= 1
        elif c == 2: i -= 1
        else: j -= 1
    return set(range(m)) - matched

fr = []
for a, b in zip(train.text_after, train.text_before):
    at, bt = tokenize(a), tokenize(b)
    fr.append(len(changed_positions(at, bt)) / max(1, len(bt)))
fr.sort()
q = lambda p: fr[int(len(fr)*p)]
print(f"changed fraction  p25 {q(.25):.3%}   median {q(.5):.3%}   p75 {q(.75):.3%}")

The median amendment touches around **1%** of its section.

Two consequences, and they drive everything:

- Whole-string similarity would be useless — copying the input through is ~99%
  correct as a string. This is why the metric scores only changed tokens.
- **Most of the answer is already in front of you.** The work is finding the one
  passage that moved and knowing what it said.

## The metric, reimplemented

Worth having locally: it is the only honest way to compare approaches, and it makes
the precision penalty concrete.

A proposed token only counts if it is the right token in the right *gap* of the shown
text -- appending plausible words at the end earns nothing.

In [ ]:
import difflib
ANCHOR = 8

def changed_in_gaps(shown_toks, other_toks):
    """Tokens of `other` not matched through from `shown`, with their GAP.

    LCS backtrace. Every unmatched token of `other` is returned as
    (gap, token), where gap = how many shown tokens precede it in the
    alignment -- i.e. which gap between two shown tokens it was inserted into.
    """
    # Long runs the two sides share (>= ANCHOR tokens, found by difflib) are
    # matched outright; the exact LCS runs only in the short stretches between
    # them. Consecutive versions share ~97% of their tokens, so this keeps a
    # full grade to about a second. Deterministic, and both the truth and the
    # prediction go through the same procedure, so the oracle scores 1.0.
    sm = difflib.SequenceMatcher(None, shown_toks, other_toks, autojunk=False)
    out, si, oi = [], 0, 0
    for a0, b0, size in sm.get_matching_blocks():
        if size < ANCHOR and size != 0:
            continue
        for g, t in _lcs_gaps(shown_toks[si:a0], other_toks[oi:b0]):
            out.append((g + si, t))
        si, oi = a0 + size, b0 + size
    return out


def _lcs_gaps(shown_toks, other_toks):
    n, m = len(shown_toks), len(other_toks)
    if m == 0:
        return []
    if n == 0:
        return [(0, t) for t in other_toks]
    prev = [0] * (m + 1)
    choices = []
    for i in range(1, n + 1):
        cur = [0] * (m + 1)
        row = bytearray(m + 1)
        si = shown_toks[i - 1]
        for j in range(1, m + 1):
            if si == other_toks[j - 1]:
                cur[j] = prev[j - 1] + 1
                row[j] = 1
            elif prev[j] >= cur[j - 1]:
                cur[j] = prev[j]
                row[j] = 2
            else:
                cur[j] = cur[j - 1]
                row[j] = 3
        choices.append(row)
        prev = cur
    out = []
    i, j = n, m
    while i > 0 and j > 0:
        c = choices[i - 1][j]
        if c == 1:
            i -= 1
            j -= 1
        elif c == 2:
            i -= 1
        else:
            out.append((i, other_toks[j - 1]))
            j -= 1
    while j > 0:
        out.append((0, other_toks[j - 1]))
        j -= 1
    return out


def score_row(prediction, shown, truth):
    s, b, p = tokenize(shown), tokenize(truth), tokenize(prediction)
    ch = changed_in_gaps(s, b)
    if not ch:
        return 1.0 if p == s else 0.0
    if not p:
        return 0.0
    changed  = Counter(ch)                       # (gap, token): right word, right place
    proposed = Counter(changed_in_gaps(s, p))
    hits = sum(min(c, proposed.get(k, 0)) for k, c in changed.items())
    if hits == 0:
        return 0.0
    rec  = hits / sum(changed.values())
    prec = hits / sum(proposed.values())
    return 2*prec*rec / (prec + rec)

# Hold out contiguous blocks of train so we can measure honestly. train.csv is in
# source order, so neighbouring rows are mostly the same CFR part -- and, where a
# section was amended repeatedly, consecutive versions of the same section. A
# random row split would put v1->v2 in train and v2->v3 in val, and the shown
# text of one IS the answer of the other. Blocks keep those together.
blk = pd.Series(range(len(train))) // 30
is_val = (blk % 5 == 0).values
tr = train[~is_val].reset_index(drop=True)
va = train[ is_val].reset_index(drop=True)
print(f"inner split: {len(tr)} train / {len(va)} val  (contiguous blocks of 30)")

def evaluate(fn, df=va):
    return sum(score_row(fn(r.text_after), r.text_after, r.text_before)
               for r in df.itertuples()) / len(df)

## Submission 1 — copy the input through

The sample submission. It is ~99% of the characters and worth ~0 of the score.

In [ ]:
print("copy-through (val):", round(evaluate(lambda t: t), 4))

pd.DataFrame({"row_id": test.row_id, "text_before": test.text_after}) \
  .to_csv(os.path.join(OUT, "submission.csv"), index=False)
print("wrote submission.csv")

## Submission 2 — substitution table + sentence locator (CPU)

Two halves, matching the two halves of the problem.

**What does an amendment do?** Align every training pair and read off the contiguous
runs that differ. Each is a substitution `(after_phrase → before_phrase)`. Keep the
ones that recur — a substitution seen once is a coincidence.

**Where did it happen?** A character n-gram classifier over sentences, trained on
whether the amendment actually touched that sentence.

In [ ]:
MIN_SUB_COUNT, MAX_SUB_TOKENS = 2, 12

def align_runs(a_toks, b_toks):
    n, m = len(a_toks), len(b_toks)
    if n == 0 or m == 0: return []
    prev = [0]*(m+1); choices = []
    for i in range(1, n+1):
        cur = [0]*(m+1); row = bytearray(m+1); ai = a_toks[i-1]
        for j in range(1, m+1):
            if ai == b_toks[j-1]: cur[j] = prev[j-1]+1; row[j] = 1
            elif prev[j] >= cur[j-1]: cur[j] = prev[j]; row[j] = 2
            else: cur[j] = cur[j-1]; row[j] = 3
        choices.append(row); prev = cur
    ops = []; i, j = n, m
    while i > 0 and j > 0:
        c = choices[i-1][j]
        if c == 1: ops.append(("=", a_toks[i-1], b_toks[j-1])); i -= 1; j -= 1
        elif c == 2: ops.append(("-", a_toks[i-1], None)); i -= 1
        else: ops.append(("+", None, b_toks[j-1])); j -= 1
    while i > 0: ops.append(("-", a_toks[i-1], None)); i -= 1
    while j > 0: ops.append(("+", None, b_toks[j-1])); j -= 1
    ops.reverse()
    runs, ca, cb = [], [], []
    for op, av, bv in ops:
        if op == "=":
            if ca or cb: runs.append((ca, cb)); ca, cb = [], []
        elif op == "-": ca.append(av)
        else: cb.append(bv)
    if ca or cb: runs.append((ca, cb))
    return runs

def mine(df):
    subs = defaultdict(Counter)
    for a, b in zip(df.text_after, df.text_before):
        for ar, br in align_runs(tokenize(a), tokenize(b)):
            if not ar or not br: continue
            if len(ar) > MAX_SUB_TOKENS or len(br) > MAX_SUB_TOKENS: continue
            subs[" ".join(ar)][" ".join(br)] += 1
    out = {}
    for ap, c in subs.items():
        bp, n = c.most_common(1)[0]
        if n >= MIN_SUB_COUNT and ap != bp: out[ap] = (bp, n)
    return out

table = mine(tr)
print(f"{len(table)} recurring substitutions")
for ap, (bp, n) in sorted(table.items(), key=lambda kv: -kv[1][1])[:8]:
    print(f"  x{n:<3} {ap[:44]!r} -> {bp[:44]!r}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

SENT = re.compile(r"(?<=[.;:])\s+")
sentences = lambda t: [s for s in SENT.split(t or "") if s.strip()]

X, y = [], []
for a, b in zip(tr.text_after, tr.text_before):
    bs = set(sentences(b))
    for s in sentences(a):
        X.append(s); y.append(0 if s in bs else 1)
print(f"{len(X)} sentences, {sum(y)} amended ({sum(y)/len(y):.1%})")

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5),
                      min_df=2, max_features=60000, sublinear_tf=True)
clf = LogisticRegression(max_iter=2000, C=2.0, random_state=SEED)
clf.fit(vec.fit_transform(X), y)
print("sentence locator trained")

In [ ]:
def predict_v2(text_after, table=table):
    sents = sentences(text_after)
    if not sents or not table: return text_after
    probs = clf.predict_proba(vec.transform(sents))[:, 1]
    for idx in sorted(range(len(sents)), key=lambda i: -probs[i]):
        s, best = sents[idx], None
        for ap, (bp, n) in table.items():
            if ap and ap in s:
                sc = n * len(ap)               # frequent AND specific
                if best is None or sc > best[0]: best = (sc, ap, bp)
        if best:
            sents[idx] = s.replace(best[1], best[2], 1)
            return " ".join(sents)
    return text_after

v2 = evaluate(predict_v2)
print("copy-through (val):", round(evaluate(lambda t: t), 4))
print("substitution+locator (val):", round(v2, 4))

### Why one edit and not many

The precision term makes over-editing expensive. Worth confirming rather than
assuming — here is the same approach applied to the top-`k` sentences.

In [ ]:
def predict_topk(text_after, k):
    sents = sentences(text_after)
    if not sents or not table: return text_after
    probs = clf.predict_proba(vec.transform(sents))[:, 1]
    done = 0
    for idx in sorted(range(len(sents)), key=lambda i: -probs[i]):
        if done >= k: break
        s, best = sents[idx], None
        for ap, (bp, n) in table.items():
            if ap and ap in s:
                sc = n * len(ap)
                if best is None or sc > best[0]: best = (sc, ap, bp)
        if best:
            sents[idx] = s.replace(best[1], best[2], 1); done += 1
    return " ".join(sents)

for k in (1, 2, 3, 5):
    print(f"  top-{k} edits (val): {evaluate(lambda t, k=k: predict_topk(t, k)):.4f}")

In [ ]:
# Refit on ALL of train, then predict the test set.
table_full = mine(train)
X, y = [], []
for a, b in zip(train.text_after, train.text_before):
    bs = set(sentences(b))
    for s in sentences(a):
        X.append(s); y.append(0 if s in bs else 1)
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5),
                      min_df=2, max_features=60000, sublinear_tf=True)
clf = LogisticRegression(max_iter=2000, C=2.0, random_state=SEED)
clf.fit(vec.fit_transform(X), y)

preds = [predict_v2(t, table_full) for t in test.text_after]
pd.DataFrame({"row_id": test.row_id, "text_before": preds}) \
  .to_csv(os.path.join(OUT, "submission.csv"), index=False)
print(f"wrote submission.csv — {sum(1 for a,p in zip(test.text_after,preds) if a!=p)}"
      f"/{len(test)} rows edited")

## Submission 3 — seq2seq over the located passage (GPU)

The table above can only replay substitutions it has literally seen. A generative
model can propose wording it has not, which is where the remaining headroom is.
An amendment usually touches several sentences, not one, so every sentence the
locator is confident about is regenerated -- not just the top one.

Sections are far longer than a comfortable encoder window, so the locator from
submission 2 earns its keep again: train the model on **passages**, not whole
sections. Each example is a window around an amended sentence, and the target is
that window's prior wording.

In [ ]:
import torch
DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
# Build passage-level pairs: a window around each sentence the amendment touched.
WINDOW = 1   # sentences of context on each side

def passage_pairs(df):
    src, tgt = [], []
    for a, b in zip(df.text_after, df.text_before):
        a_s, b_s = sentences(a), sentences(b)
        bset = set(b_s)
        for i, s in enumerate(a_s):
            if s in bset:
                continue                       # untouched, nothing to learn
            lo, hi = max(0, i-WINDOW), min(len(a_s), i+WINDOW+1)
            after_win = " ".join(a_s[lo:hi])
            # nearest-length prior sentence not present in `after` = its counterpart
            cands = [x for x in b_s if x not in set(a_s)]
            if not cands:
                continue
            best = min(cands, key=lambda x: abs(len(x) - len(s)))
            src.append(after_win); tgt.append(best)
    return src, tgt

src_tr, tgt_tr = passage_pairs(tr)
print(f"{len(src_tr)} passage pairs from {len(tr)} sections")
if src_tr:
    print("\nSRC:", src_tr[0][:220])
    print("TGT:", tgt_tr[0][:220])

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = "google/flan-t5-small"
MAX_SRC, MAX_TGT = 320, 160
EPOCHS, BATCH, LR = 3, 8, 3e-4

torch.manual_seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(DEVICE)

PROMPT = "Restore the wording of this regulation before it was amended: "

def batches(src, tgt, bs):
    for i in range(0, len(src), bs):
        yield src[i:i+bs], tgt[i:i+bs]

opt = torch.optim.AdamW(model.parameters(), lr=LR)
model.train()
for ep in range(EPOCHS):
    tot = n = 0
    for bs_src, bs_tgt in batches(src_tr, tgt_tr, BATCH):
        enc = tok([PROMPT+s for s in bs_src], max_length=MAX_SRC,
                  truncation=True, padding=True, return_tensors="pt").to(DEVICE)
        lab = tok(bs_tgt, max_length=MAX_TGT, truncation=True,
                  padding=True, return_tensors="pt").input_ids.to(DEVICE)
        lab[lab == tok.pad_token_id] = -100
        loss = model(**enc, labels=lab).loss
        loss.backward(); opt.step(); opt.zero_grad()
        tot += loss.item(); n += 1
    print(f"epoch {ep+1}: loss {tot/max(1,n):.4f}")

In [ ]:
MIN_PROB, MAX_EDITS = 0.5, 8

@torch.no_grad()
def predict_v3(text_after):
    """Regenerate every sentence the locator is confident about, splice back."""
    sents = sentences(text_after)
    if not sents:
        return text_after
    probs = clf.predict_proba(vec.transform(sents))[:, 1]
    idxs = [i for i in sorted(range(len(sents)), key=lambda i: -probs[i])
            if probs[i] >= MIN_PROB][:MAX_EDITS]
    if not idxs:
        return text_after
    windows = [" ".join(sents[max(0, i-WINDOW):i+WINDOW+1]) for i in idxs]
    enc = tok([PROMPT+w for w in windows], max_length=MAX_SRC, truncation=True,
              padding=True, return_tensors="pt").to(DEVICE)
    out = model.generate(**enc, max_new_tokens=MAX_TGT, num_beams=1,
                         do_sample=False)
    for i, g in zip(idxs, tok.batch_decode(out, skip_special_tokens=True)):
        g = g.strip()
        if g:
            sents[i] = g
    return " ".join(sents)

model.eval()
sample = va.head(40)
v3 = sum(score_row(predict_v3(r.text_after), r.text_after, r.text_before)
         for r in sample.itertuples()) / len(sample)
v2s = sum(score_row(predict_v2(r.text_after), r.text_after, r.text_before)
          for r in sample.itertuples()) / len(sample)
print(f"on {len(sample)} val rows —  seq2seq {v3:.4f}   vs   table {v2s:.4f}")

In [ ]:
# Ship whichever actually won on validation. Measured, not assumed.
use_seq2seq = v3 > v2s
print("shipping:", "seq2seq" if use_seq2seq else "substitution table")

fn = predict_v3 if use_seq2seq else (lambda t: predict_v2(t, table_full))
preds = [fn(t) for t in test.text_after]
pd.DataFrame({"row_id": test.row_id, "text_before": preds}) \
  .to_csv(os.path.join(OUT, "submission.csv"), index=False)
print(f"wrote submission.csv — {len(preds)} rows")

## What to try next

- **Better localisation.** Everything downstream depends on finding the amended
  sentence. A cross-encoder over (section, sentence) pairs would beat char n-grams.
- **Predict the change *type* first.** Deadlines, dollar thresholds, defined terms
  and cross-references each move in characteristic ways; a model that classifies the
  edit before generating it has a much narrower job.
- **Condition on the section's own subject.** No title, part or section number is
  published (the section's own number is masked as `[SECTION]`), but the heading and
  the cross-references in the text still say what area a section governs, and
  different regulatory areas amend different things.
- **Respect the precision term.** The top-`k` sweep showed what over-editing costs.
  Any approach that rewrites confidently in several places needs to be measured
  against a one-edit variant before shipping.